# Module 1.2: Tensors & Linear Algebra

If you haven't touched math in years, don't worry! This notebook is designed to build your skills from the ground up.
Transformers speak the language of **Linear Algebra**, but at its core, it's just about organizing and comparing numbers.

> 📎 **Builds on the Math Primer (Module 1.1).** This notebook *applies* three
> things you just met there: the summation symbol $\sum$, the **dot product**
> ($\mathbf{a}\cdot\mathbf{b}=\|\mathbf a\|\|\mathbf b\|\cos\theta$, "similarity by
> angle"), and the exponential $e^x$. If any of those feel shaky, flip back — this
> page assumes them.

## 1. What is a Vector and a Matrix?

### The "Fruit Feature" Analogy
Imagine you are describing a **Fruit** to a computer.

| Feature | Value |
| :--- | :--- |
| **Sweetness** | 0.9 |
| **Crunchiness**| 0.8 |
| **Color (Red)**| 0.2 |

- **Vector**: A single list of numbers (a word). `[0.9, 0.8, 0.2]` is the word's "numerical identity".
- **Matrix**: A grid of vectors (a sentence). A 3-word sentence is a 3-row matrix.
- **Tensor**: A stack of matrices (a batch of sentences).

**The Output**: Numbers that the computer can use for geometry and similarity.

In [12]:
import torch

# Reproducibility: fix the random seed so every run gives the same numbers.
torch.manual_seed(0)

# A Vector (3 features)
word_vector = torch.tensor([0.9, 0.8, 0.2])

# A Matrix (2 words, 3 features)
sentence_matrix = torch.tensor([[0.9, 0.8, 0.2], [0.1, 0.2, 0.8]])  # Word 1  # Word 2

print(f"Vector shape: {word_vector.shape}")
print(f"Matrix shape (Words, Features): {sentence_matrix.shape}")

Vector shape: torch.Size([3])
Matrix shape (Words, Features): torch.Size([2, 3])


## 2. Transpose (Alignment)

### The Concept
Transposing **flips a matrix over its diagonal**: every row becomes a column and
every column becomes a row. It is written $\mathbf{A}^T$. A $2\times 3$ matrix
becomes a $3\times 2$ one — the numbers don't change, only their positions:

$$\begin{bmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \end{bmatrix}^{T}
= \begin{bmatrix} 1 & 4 \\ 2 & 5 \\ 3 & 6 \end{bmatrix}$$

Read it as: *"row 1 `[1 2 3]` stands up to become column 1."*

### Why?
Matrix multiplication only lines up when the **inner dimensions match** (section 4
below). To compare a batch of Queries against a batch of Keys, we transpose the
Keys so their feature axis "docks" against the Queries' feature axis — turning
$(T, d)\,@\,(T, d)$ (illegal, inner dims $d$ and $T$ don't match) into
$(T, d)\,@\,(d, T)$ (legal), which yields a $T\times T$ grid of every-word-vs-every-word
scores. That single transpose is what makes attention's score matrix possible.

In [13]:
A = torch.tensor([[1, 2, 3], [4, 5, 6]])
print("Original (2x3):\n", A)
print("Transposed (3x2):\n", A.T)

Original (2x3):
 tensor([[1, 2, 3],
        [4, 5, 6]])
Transposed (3x2):
 tensor([[1, 4],
        [2, 5],
        [3, 6]])


## 3. Dot Product (Similarity Score)

### The Math
$$\mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^{n} a_i b_i$$

### Why?
It measures **how much two words overlap**. 
- **High Score**: Similar meaning (e.g., 'King' and 'Royal').
- **Zero Score**: Unrelated (e.g., 'King' and 'Banana').
- **Negative Score**: Opposite (rare in basic attention, but possible).

In [14]:
v1 = torch.tensor([1.0, 1.0])
v2 = torch.tensor([1.0, 1.0])  # Same direction  -> positive (aligned)
v3 = torch.tensor([-1.0, 1.0])  # Perpendicular   -> zero (unrelated)
v4 = torch.tensor([-1.0, -1.0])  # Opposite        -> negative (anti-aligned)

print(f"Agreement Score (same direction): {torch.dot(v1, v2)}")  # 1*1 + 1*1 = 2
print(f"Zero Score (perpendicular):       {torch.dot(v1, v3)}")  # 1*-1 + 1*1 = 0
print(f"Negative Score (opposite):        {torch.dot(v1, v4)}")  # 1*-1 + 1*-1 = -2

Agreement Score (same direction): 2.0
Zero Score (perpendicular):       0.0
Negative Score (opposite):        -2.0


## 4. Matrix Multiplication (Parallel Efficiency)

### Tracking Shapes (CRITICAL 🔥)
In LLMs, we process hundreds of words at once. 
- $Q$: (2 words, 3 features)
- $K^T$: (3 features, 2 words)
- **Result**: (2, 2) grid of similarities.

**The Output**: A matrix where every entry $(i, j)$ is the similarity of word $i$ with word $j$.

In [15]:
Q = torch.randn(2, 3)
K_T = torch.randn(3, 2)
scores = torch.matmul(Q, K_T)
print(f"Similarity Matrix Shape: {scores.shape}")

Similarity Matrix Shape: torch.Size([2, 2])


### How is each number computed? (row · column)

Matrix multiplication looks fancy, but it is built *entirely* out of the dot
product from Module 1.1. To fill in the entry at **row $i$, column $j$** of the
result, you take **row $i$ of the left matrix** and **dot it with column $j$ of
the right matrix**:

$$C_{ij} \;=\; \sum_k A_{ik}\,B_{kj}$$

Read it out loud: *"entry $(i,j)$ of $C$ is the dot product of row $i$ of $A$ with
column $j$ of $B$."* That is also **why the inner dimensions must match** — a row
and a column can only be dotted together if they are the same length.

Let's compute one by hand and check it against `@`.

> **In a Transformer this exact multiply runs on 4-D tensors**, batched over every
> sentence and head at once — the $(B,H,T,d)\,@\,(B,H,d,T)$ shape you met in
> **Module 0.1, section 6**. The per-entry math is identical; there are just many
> copies of it happening in parallel.

In [16]:
A = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])  # shape (2, 3)  -> 2 rows
B = torch.tensor([[7.0, 8.0], [9.0, 10.0], [11.0, 12.0]])  # shape (3, 2)  -> 2 columns

# Fill the (2, 2) result one entry at a time: each is a row·column dot product.
C = torch.zeros(2, 2)
for i in range(2):  # for each row of A
    for j in range(2):  # for each column of B
        C[i, j] = torch.dot(A[i, :], B[:, j])
        print(
            f"C[{i},{j}] = row{i} · col{j} = {A[i].tolist()} · {B[:, j].tolist()} = {C[i, j]:.0f}"
        )

print("\nBy hand:\n", C)
print("torch A @ B:\n", A @ B)
print("match:", torch.allclose(C, A @ B))

C[0,0] = row0 · col0 = [1.0, 2.0, 3.0] · [7.0, 9.0, 11.0] = 58
C[0,1] = row0 · col1 = [1.0, 2.0, 3.0] · [8.0, 10.0, 12.0] = 64
C[1,0] = row1 · col0 = [4.0, 5.0, 6.0] · [7.0, 9.0, 11.0] = 139
C[1,1] = row1 · col1 = [4.0, 5.0, 6.0] · [8.0, 10.0, 12.0] = 154

By hand:
 tensor([[ 58.,  64.],
        [139., 154.]])
torch A @ B:
 tensor([[ 58.,  64.],
        [139., 154.]])
match: True


## 5. Softmax (Focusing the Model)

### Math
$$\sigma(\mathbf{z})_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

### Why softmax — and why not just divide by the sum?
We need to turn raw scores into **weights that are all positive and add up to 1**
(a probability distribution — Module 1.1). So why not simply divide each score by
the total? Two reasons:

1. **Scores can be negative.** Plain division by the sum can produce negative or
   exploding weights. $e^{z}$ is **always positive** (Module 1.1), so every weight
   is valid.
2. **We want *focus*, not a flat average.** $e^{z}$ grows fast, so it
   **exaggerates** the gap between the top score and the rest — the winner takes a
   big share and near-ties get pushed apart. That "pick a clear winner" behavior is
   exactly what attention needs.

In [17]:
raw = torch.tensor([10.0, 1.0, 0.1])
probs = torch.softmax(raw, dim=0)
print(f"Attention Focus: {probs}")
print(f"Sums to: {probs.sum():.4f}  (softmax outputs always sum to 1)")

Attention Focus: tensor([9.9983e-01, 1.2339e-04, 5.0166e-05])
Sums to: 1.0000  (softmax outputs always sum to 1)


### Softmax, computed step by step

The formula $\sigma(\mathbf z)_i = \dfrac{e^{z_i}}{\sum_j e^{z_j}}$ is only **two
moves**, and both use the exponential from Module 1.1:

1. **Exponentiate** every score: $e^{z_i}$. (Recall $e^x$ is always positive — so
   every result is a valid, non-negative weight.)
2. **Normalize**: divide each by the total $\sum_j e^{z_j}$, so they add up to 1.

Here it is with actual numbers, by hand, next to `torch.softmax`.

In [18]:
z = torch.tensor([2.0, 1.0, 0.1])

exps = torch.exp(z)  # step 1: e^z for each score (all positive)
probs = exps / exps.sum()  # step 2: divide by the total so they sum to 1

print("logits z         :", [round(v, 3) for v in z.tolist()])
print("step 1  e^z      :", [round(v, 3) for v in exps.tolist()])
print("        sum(e^z) :", round(exps.sum().item(), 3))
print(
    "step 2  e^z / sum:",
    [round(v, 3) for v in probs.tolist()],
    " (sum =",
    round(probs.sum().item(), 2),
    ")",
)
print("torch.softmax    :", [round(v, 3) for v in torch.softmax(z, dim=0).tolist()])
print("\nSame numbers -- softmax is just 'exponentiate, then normalize'.")

logits z         : [2.0, 1.0, 0.1]
step 1  e^z      : [7.389, 2.718, 1.105]
        sum(e^z) : 11.213
step 2  e^z / sum: [0.659, 0.242, 0.099]  (sum = 1.0 )
torch.softmax    : [0.659, 0.242, 0.099]

Same numbers -- softmax is just 'exponentiate, then normalize'.


### One safety trick: subtract the max first

$e^{z}$ grows *explosively* — $e^{1000}$ overflows to infinity on a computer, and
`inf / inf` is `nan`. The fix used inside every real softmax: **subtract the
largest score from all of them before exponentiating.** Shifting every term by the
same amount leaves the result **identical**, but now the biggest exponent is
$e^{0}=1$, so nothing overflows.

In [19]:
z = torch.tensor([1000.0, 1001.0, 1002.0])  # huge scores -> e^z would overflow

naive = torch.exp(z) / torch.exp(z).sum()  # inf / inf = nan
stable = torch.exp(z - z.max()) / torch.exp(z - z.max()).sum()  # shift by the max

print("naive  (overflows):", naive)
print("stable (shifted)  :", stable)
print("torch.softmax     :", torch.softmax(z, dim=0), " <- does the shift for you")
print("\nSame distribution, no infinities -- which is why you call torch.softmax,")
print("never a hand-rolled exp/sum, in real code.")

naive  (overflows): tensor([nan, nan, nan])
stable (shifted)  : tensor([0.0900, 0.2447, 0.6652])
torch.softmax     : tensor([0.0900, 0.2447, 0.6652])  <- does the shift for you

Same distribution, no infinities -- which is why you call torch.softmax,
never a hand-rolled exp/sum, in real code.


## 6. Numerical Stability (the √dₖ trick)

### What is $d_k$?
$d_k$ is the **dimension of the Query/Key vectors** — how many features each Query
and Key has. A score is a dot product, which **sums $d_k$ products together**
(Module 1.1). Add up more terms and the total tends to grow larger in magnitude —
so a bigger $d_k$ means bigger, more spread-out scores, purely from summing more
terms.

### Why huge scores are a problem — you can see this *without any calculus*
Feed very large, spread-out scores into softmax and it becomes **winner-take-all**:
one weight rushes to ≈1 and the rest collapse to ≈0 — a "peaky", almost one-hot
distribution. Attention then fixates on a single token and effectively ignores the
others, even ones that mattered. The demo below shows it directly: the *same*
scores multiplied by 10 give a far peakier softmax.

### And why that also stalls learning — a short peek ahead to Module 1.4
There is a second, deeper cost. Once softmax is saturated (peaky), nudging the
scores barely moves its output — so the learning signal (the *gradient*, which you
build by hand in **Module 1.4**) is nearly zero and the model can hardly improve.
You don't need gradients yet to use the fix; just keep the phrase *"saturated
softmax = hard to learn from."*

### The fix
Divide the scores by $\sqrt{d_k}$ **before** softmax. This pulls their magnitude
back to a sane range (variance ≈ 1 — Module 1.1), keeping softmax responsive
instead of saturated. The demo prints the variance before and after.

In [20]:
# --- Demo A: big, spread-out logits SATURATE softmax (peaky, near one-hot) ---
big = torch.tensor([10.0, 1.0, 0.1])  # large gap between the top score and the rest
small = torch.tensor([1.0, 0.1, 0.01])  # same ordering, but scores are close together

print("softmax(big logits)  :", torch.softmax(big, dim=0))
print("  -> one value ~1, others ~0  (SATURATED: gradient here is ~0, hard to learn)\n")

print("softmax(small logits):", torch.softmax(small, dim=0))
print("  -> probabilities stay closer together (responsive: gradients still flow)\n")

# --- Demo B: dividing by sqrt(d_k) tames the variance of the scores ---
torch.manual_seed(0)
d_k = 64
Q = torch.randn(1000, d_k)  # 1000 random Query vectors
K = torch.randn(1000, d_k)  # 1000 random Key vectors
raw_scores = (Q * K).sum(dim=1)  # dot products WITHOUT scaling
scaled_scores = raw_scores / (d_k**0.5)  # dot products WITH /sqrt(d_k)

print(f"d_k = {d_k}")
print(f"Variance of scores BEFORE scaling: {raw_scores.var():.2f}  (grows with d_k)")
print(
    f"Variance of scores AFTER  /sqrt(d_k): {scaled_scores.var():.2f}  (~1, well-behaved)"
)

softmax(big logits)  : tensor([9.9983e-01, 1.2339e-04, 5.0166e-05])
  -> one value ~1, others ~0  (SATURATED: gradient here is ~0, hard to learn)

softmax(small logits): tensor([0.5624, 0.2286, 0.2090])
  -> probabilities stay closer together (responsive: gradients still flow)

d_k = 64
Variance of scores BEFORE scaling: 64.40  (grows with d_k)
Variance of scores AFTER  /sqrt(d_k): 1.01  (~1, well-behaved)


## 7. The Final Boss: Scaled Dot-Product Attention

$$ Attention(Q, K, V) = softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$

### Library Analogy (QKV)
- **Query ($Q$)**: The book title you are searching for.
- **Key ($K$)**: The label on the spine of the book.
- **Value ($V$)**: The actual knowledge inside the book.

**The Output**: A new vector that is a **Weighted Average** (Expectation) of the values based on how well the query matched the keys.

> 🔭 **A preview, not the full lesson.** The cell below jumps ahead and shows *attention* — the star of this whole course — because it's the payoff of the math on this page: every line is nothing but the dot products, matrix multiplications, and softmax you just learned. Don't worry about memorizing it. You'll rebuild it slowly, and understand every design choice, in **Module 3.1**.

In [21]:
import torch.nn.functional as F
from math import sqrt


def scaled_dot_product_attention(query, key, value, mask=None):
    dk = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / sqrt(dk)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, value)
    return output, attention_weights


torch.manual_seed(0)
x = torch.randn(1, 3, 4)  # 1 batch, 3 words, 4 features each

# Passing the SAME x as Q, K, and V is what makes this *self*-attention:
# every word builds its Query, Key, and Value from itself and attends over the same sequence.
output, weights = scaled_dot_product_attention(x, x, x)

print("Attention weights (rows = each word's focus over all 3 words):")
print(weights.squeeze(0))
print("\nEach row sums to 1:", weights.squeeze(0).sum(dim=-1))
print("\nFinal Output Layer Shape:", output.shape)

Attention weights (rows = each word's focus over all 3 words):
tensor([[0.9629, 0.0097, 0.0274],
        [0.0449, 0.7616, 0.1934],
        [0.2556, 0.3896, 0.3548]])

Each row sums to 1: tensor([1.0000, 1.0000, 1.0000])

Final Output Layer Shape: torch.Size([1, 3, 4])


### 🏋️ Try it yourself

1. **Word similarity by hand.** Create three 4-dimensional word vectors of your own (e.g. `king`, `queen`, `banana`). Use `torch.dot` to compute all three pairwise dot products and check whether the two "royal" words score higher with each other than either does with `banana`.
2. **Watch softmax saturate.** Take the logits `[2.0, 1.0, 0.5]`, run softmax, then *multiply the same logits by 10* and run softmax again. Print both results and describe in a comment how the distribution changes (does it get peakier or flatter?).

In [22]:
import torch

# --- Exercise 1: word similarity ---
king = torch.tensor([0.9, 0.8, 0.1, 0.7])
queen = torch.tensor([0.8, 0.9, 0.1, 0.6])
banana = torch.tensor([0.1, 0.0, 0.9, 0.2])

# TODO: print torch.dot(king, queen), torch.dot(king, banana), torch.dot(queen, banana)
# Is king-queen the highest?


# --- Exercise 2: softmax saturation ---
logits = torch.tensor([2.0, 1.0, 0.5])
# TODO: print torch.softmax(logits, dim=0)
# TODO: print torch.softmax(logits * 10, dim=0)
# Comment: peakier or flatter?